# Gold: dim_date

Run-once only. Just tweak the start and end dates if you need a different range.

## Import Helper Functions

In [1]:
from src.config_loader import load_config
from src.spark_sql_magic import sql
from src.gold.dims.date import build_dim_date
from src.writers import overwrite_table

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


## Load Configs

In [2]:
cfg = load_config()

JOB_NAMES = cfg["spark_jobs"]["jobs"]
CATALOG = cfg["general"]["catalog"]
SILVER_NAMESPACE = cfg["general"]["namespaces"]["silver"]
GOLD_NAMESPACE = cfg["general"]["namespaces"]["gold"]

TARGET_TABLE = cfg["gold"]["dim_date"]["target_table"]
START_DATE = cfg["gold"]["dim_date"]["start_date"]
END_DATE = cfg["gold"]["dim_date"]["end_date"]

## Import Libraries and Start Session

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window as W
import pyspark
import datetime
import json

spark = (
    SparkSession.builder
        .appName(JOB_NAMES["gold_date"])
        .getOrCreate()
)

## Run Pipeline

In [4]:
def run_dim_date_pipeline(spark):

    df = build_dim_date(spark, START_DATE, END_DATE)

    overwrite_table(df, TARGET_TABLE)

In [5]:
# if __name__ == "__main__":
#     from pyspark.sql import SparkSession

#     spark = SparkSession.builder.getOrCreate()
run_dim_date_pipeline(spark)

## Sanity Check

In [6]:
%%sql
SHOW TABLES IN polaris.gold;

+---------+------------------+-----------+
|namespace|tableName         |isTemporary|
+---------+------------------+-----------+
|gold     |dim_customers_scd2|false      |
|gold     |dim_sellers_scd2  |false      |
|gold     |dim_products_scd2 |false      |
|gold     |dim_date          |false      |
+---------+------------------+-----------+



In [7]:
%%sql
SELECT * FROM polaris.gold.dim_date
LIMIT 10

[Stage 1:>                                                          (0 + 1) / 1]

+----------+--------+----+-------+-----+---+-----------+---------+------------+----------+
|ds        |date_sk |year|quarter|month|day|day_of_week|day_name |week_of_year|is_weekend|
+----------+--------+----+-------+-----+---+-----------+---------+------------+----------+
|2010-01-01|20100101|2010|1      |1    |1  |6          |Friday   |53          |false     |
|2010-01-02|20100102|2010|1      |1    |2  |7          |Saturday |53          |true      |
|2010-01-03|20100103|2010|1      |1    |3  |1          |Sunday   |53          |true      |
|2010-01-04|20100104|2010|1      |1    |4  |2          |Monday   |1           |false     |
|2010-01-05|20100105|2010|1      |1    |5  |3          |Tuesday  |1           |false     |
|2010-01-06|20100106|2010|1      |1    |6  |4          |Wednesday|1           |false     |
|2010-01-07|20100107|2010|1      |1    |7  |5          |Thursday |1           |false     |
|2010-01-08|20100108|2010|1      |1    |8  |6          |Friday   |1           |false     |